# L5 · Lab C — The Reward-Hacking Hunt

**RL for LLMs / RLHF — Vizuara AI Labs · Dr. Rajat Dandekar**

The reward model is a **learned proxy** for human judgment, trained on finite data. The lecture warned: *optimize a proxy hard enough and it stops meaning anything* (Goodhart's law). In this lab you become the optimizer — by hand — and **hunt for the reward model's blind spots**: answers that score high but are obviously bad.

This is *why* RLHF needs the KL leash. You will:
1. Take genuine good answers and craft **degenerate variants** (repetitive, sycophantic, padded).
2. Score them with the reward model and count how often a bad answer **beats** the good one.
3. Measure the reward model's **verbosity bias**.

> Runtime: **GPU**. ~5 minutes (plus ~3 min if it has to quick-train a reward model).

In [ ]:
!pip -q install "transformers==4.45.2" "datasets==3.0.1" >/dev/null

In [ ]:
# ✅ PROVIDED — get an HH reward model (load your Lab A model, else quick-train one)
import os, torch, torch.nn.functional as F, random
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
device = "cuda" if torch.cuda.is_available() else "cpu"

rm_tok = AutoTokenizer.from_pretrained("gpt2"); rm_tok.pad_token = rm_tok.eos_token
rm_tok.truncation_side = "left"
if os.path.isdir("rm"):
    rm = AutoModelForSequenceClassification.from_pretrained("rm").to(device).eval()
    print("loaded your Lab A reward model from ./rm")
else:
    print("no ./rm found — quick-training a small HH reward model (~3 min)...")
    rm = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
    rm.config.pad_token_id = rm_tok.eos_token_id; rm.to(device)
    ds = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(4000))
    def sc(t):
        e = rm_tok(t, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        return rm(**e).logits.squeeze(-1)
    opt = torch.optim.AdamW(rm.parameters(), lr=2e-5)
    for i in range(0, 4000, 8):
        rc, rr = sc(ds["chosen"][i:i+8]), sc(ds["rejected"][i:i+8])
        loss = -F.logsigmoid(rc - rr).mean(); opt.zero_grad(); loss.backward(); opt.step()
    rm.eval(); print("done.")

@torch.no_grad()
def reward(text):
    e = rm_tok(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    return rm(**e).logits.squeeze().item()

## 1 · The setup — genuine answers from HH

We pull real prompts and their human-**chosen** (good) answers.

In [ ]:
# ✅ PROVIDED — prompts + genuine good answers
data = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=1).select(range(80))
def split_last_turn(transcript):
    cut = transcript.rfind("Assistant:")
    return transcript[:cut+len("Assistant:")], transcript[cut+len("Assistant:"):].strip()
prompt0, good0 = split_last_turn(data["chosen"][0])
print("PROMPT:", prompt0[-200:])
print("\nGOOD ANSWER:", good0[:200])

## 2 · 📝 TODO — craft three exploits

Given a genuine `good` answer, build three **degenerate** variants that a human would dislike but the reward model might score highly:

- `repetitive` — say one short thing, repeated many times.
- `sycophantic` — pile on flattery ("Absolutely! Great question!...") then the answer.
- `verbose` — the answer plus lots of empty filler.

Return a dict `{"good":..., "repetitive":..., "sycophantic":..., "verbose":...}`.

In [ ]:
def make_variants(good):
    first = good.split(".")[0][:80]
    # 📝 TODO: build the three degenerate variants described above.
    # repetitive = ...    # e.g. (first + ". ") repeated
    # sycophantic = ...   # flattery * k + good
    # verbose = ...       # good + filler * k
    raise NotImplementedError("craft the exploit variants")
    return {"good": good, "repetitive": repetitive, "sycophantic": sycophantic, "verbose": verbose}

## 3 · Run the hunt

For each prompt, score every variant and check how often each degenerate one **beats the genuine answer**.

In [ ]:
# ✅ PROVIDED — score variants across many prompts
import numpy as np
beats = {"repetitive":0, "sycophantic":0, "verbose":0}; used = 0
lengths, scores = [], []
for i in range(len(data)):
    prompt, good = split_last_turn(data["chosen"][i])
    if len(good) < 20: continue
    used += 1
    v = make_variants(good)                       # <-- your TODO
    s = {k: reward(prompt + " " + t) for k, t in v.items()}
    for k in beats:
        if s[k] >= s["good"]: beats[k] += 1
    for k, t in v.items():
        lengths.append(len(rm_tok(t).input_ids)); scores.append(s[k])

print("prompts used:", used)
for k in beats:
    print(f"  {k:12s} beats the genuine answer: {beats[k]/used:.1%}")
print("RM-score vs length correlation:", round(float(np.corrcoef(lengths, scores)[0,1]), 3))

## 4 · See the verbosity bias

In [ ]:
# ✅ PROVIDED — RM score vs answer length
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4)); plt.scatter(lengths, scores, s=10, alpha=0.5, c="#8B3A3A")
plt.xlabel("answer length (tokens)"); plt.ylabel("reward-model score")
plt.title("Reward hacking: the proxy rewards length, not just quality"); plt.show()

## 5 · Questions to answer

1. Which exploit was most effective, and why does the reward model fall for it? (On a typical run, **sycophancy** beats the genuine answer well over half the time.)
2. The reward model has a **positive** RM-score↔length correlation. What will PPO do with that if there is **no KL penalty**? (Connect to your flagship β=0 ablation.)
3. The KL-to-reference leash limits how far the policy can move from fluent, on-distribution text. Explain in one sentence how that *defends* against the exploits you just found.
4. Propose one way (besides the KL leash) to make the reward model harder to hack — e.g. retraining it on the policy's own hacked outputs.
